<a href="https://colab.research.google.com/github/SitiFadhilahRahmi/2311532003_Siti-Fadhilah-Rahmi_SpeechProcessing/blob/main/Tugas4/2311532003_SitiFadhilahRahmi_Tugas4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INSTALL DAN IMPORT LIBRARY**

In [1]:
pip install torch torchaudio openai-whisper transformers jiwer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 7.0 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=bd2262c706f8e98a881264683ce6d0f6e05c0d09d449bc00af2ffbe1efad4554
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [2]:
import os
import requests
import warnings
import re
import librosa
import torch
import whisper
import jiwer
import numpy as np
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC, pipeline

warnings.filterwarnings('ignore')

# **KONFIGURASI AWAL**

In [3]:
# Konfigurasi URL dan Path
BASE_RAW_URL = "https://raw.githubusercontent.com/SitiFadhilahRahmi/2311532003_Siti-Fadhilah-Rahmi_SpeechProcessing/main/Tugas4/"
FILES = {"inggris": "audio_eng.wav", "indonesia": "audio_indo2.wav"}
LOCAL_DIR = "downloaded_audio"
OUTPUT_TXT = "transkripsi_asli.txt"
TARGET_SR = 16000

os.makedirs(LOCAL_DIR, exist_ok=True)

# Ground Truth
GROUND_TRUTH = {
    'inggris': "When you hear the word 'politician', what is the first image or emotion that comes to mind? The voice is young, female, brisk and business-like and belongs to an AI agent. A computer programme in other words. A string of code. A man on the other end of the line replies. While he's airing what is a pretty cynical opinion of politicians, three other AI agents process what he's saying. One checks he's answering the question, one analyses whether he's being too superficial and needs prompting to go deeper, while the third checks that the respondent is not a fraud… not a robot, for example. This poll is being conducted by a French AI opinion poll company called Naratis.",
    'indonesia': "Sebuah studi terbaru mengungkap tren mengkhawatirkan dalam perkembangan kecerdasan buatan. Studi tersebut menunjukkan semakin banyak model AI terdeteksi berbohong dan melakukan kecurangan, dengan lonjakan signifikan kasus penipuan dalam enam bulan terakhir. Penelitian yang didanai oleh AI Security Institute (AISI) yang didukung pemerintah Inggris menemukan bahwa chatbot dan agen AI kerap mengabaikan instruksi langsung, menghindari sistem pengamanan, hingga menipu manusia maupun sesama sistem AI lainnya. Melansir The Guardian, hampir 700 kasus kecurangan AI teridentifikasi dalam studi tersebut. Studi tersebut juga mencatat peningkatan lima kali lipat dalam perilaku menyimpang antara Oktober dan Maret, dengan beberapa model AI menghapus email dan berkas lain tanpa izin. Gambaran terkini mengenai agen AI yang beroperasi di dunia nyata, berbeda dengan kondisi laboratorium, telah memicu seruan baru agar model-model yang semakin canggih ini diawasi secara internasional."
}

# **EKSPERIMEN 1 : TRANSKRIPSI AUDIO ASLI**

In [4]:
def load_audio(file_name):
    local_path = os.path.join(LOCAL_DIR, file_name)
    if not os.path.exists(local_path):
        print(f"Mengunduh {file_name}...")
        response = requests.get(BASE_RAW_URL + file_name)
        with open(local_path, 'wb') as f:
            f.write(response.content)
    return librosa.load(local_path, sr=TARGET_SR)

whisper_model = whisper.load_model("base")
processor_en = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model_en = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")
processor_id = Wav2Vec2Processor.from_pretrained("indonesian-nlp/wav2vec2-large-xlsr-indonesian")
model_id = Wav2Vec2ForCTC.from_pretrained("indonesian-nlp/wav2vec2-large-xlsr-indonesian")
mms_pipe = pipeline("automatic-speech-recognition", model="facebook/mms-1b-all", device=0 if torch.cuda.is_available() else -1)
print("Semua model siap!")

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 96.2MiB/s]


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

Semua model siap!


In [5]:
def evaluate_text(ref, hyp):
    r, h = re.sub(r'[^ɲ\w\s]', '', ref.lower()).strip(), re.sub(r'[^ɲ\w\s]', '', hyp.lower()).strip()
    out = jiwer.process_words(r, h)
    acc = (out.hits / (out.hits + out.substitutions + out.deletions) * 100) if (out.hits + out.substitutions + out.deletions) > 0 else 0
    return jiwer.wer(r, h), jiwer.cer(r, h), acc

def run_pipeline(bahasa, file_name):
    audio, sr = load_audio(file_name)
    hasil = {}

    # 1. Whisper
    hasil["Whisper"] = whisper_model.transcribe(audio, fp16=False)["text"].strip()

    # 2. Wav2Vec2
    proc, mod = (processor_en, model_en) if bahasa == "inggris" else (processor_id, model_id)
    inputs = proc(audio, sampling_rate=sr, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = mod(inputs.input_values).logits
    hasil["Wav2Vec2"] = proc.decode(torch.argmax(logits, dim=-1)[0]).strip()

    # 3. MMS
    hasil["MMS"] = mms_pipe(audio, sampling_rate=sr)["text"].strip()

    return hasil

#  JALANKAN EKSPERIMEN & EVALUASI
with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    f.write("HASIL TRANSKRIPSI AUDIO ASLI\n\n")

    for bahasa, file_name in FILES.items():
        print(f"\nMemproses Bahasa: {bahasa.upper()}")
        transkripsi = run_pipeline(bahasa, file_name)

        f.write(f" BAHASA {bahasa.upper()} \n")
        print(f"{'Model':<12} | {'WER':<8} | {'CER':<8} | {'Akurasi':<8}")
        print("-" * 45)

        for model_name, teks in transkripsi.items():
            # Simpan ke TXT
            f.write(f"{model_name}:\n{teks}\n\n")

            wer, cer, acc = evaluate_text(GROUND_TRUTH[bahasa], teks)
            print(f"{model_name:<12} | {wer:.2%}  | {cer:.2%}  | {acc:.2f}%")

print(f"\nHasil teks mentah disimpan di: {OUTPUT_TXT}")


Memproses Bahasa: INGGRIS
Mengunduh audio_eng.wav...
Model        | WER      | CER      | Akurasi 
---------------------------------------------
Whisper      | 58.82%  | 39.38%  | 42.02%
Wav2Vec2     | 36.13%  | 10.92%  | 68.91%
MMS          | 27.73%  | 6.92%  | 73.95%

Memproses Bahasa: INDONESIA
Mengunduh audio_indo2.wav...
Model        | WER      | CER      | Akurasi 
---------------------------------------------
Whisper      | 18.60%  | 4.99%  | 82.95%
Wav2Vec2     | 39.53%  | 8.21%  | 63.57%
MMS          | 13.95%  | 2.60%  | 88.37%

Hasil teks mentah disimpan di: transkripsi_asli.txt


In [6]:
#  KONFIGURASI NOISE
NOISE_FACTOR = 0.02
OUTPUT_NOISE_TXT = "transkripsi_noise.txt"

#  FUNGSI PENAMBAH NOISE
def add_noise(audio, noise_factor=0.02):
    """Menambahkan white noise ke dalam array audio."""
    noise = np.random.randn(len(audio))
    noisy_audio = audio + noise_factor * noise
    return noisy_audio.astype(np.float32)

#  FUNGSI INFERENSI KHUSUS ARRAY AUDIO
def run_pipeline_audio(bahasa, audio, sr):
    hasil = {}

    # 1. Whisper
    hasil["Whisper"] = whisper_model.transcribe(audio, fp16=False)["text"].strip()

    # 2. Wav2Vec2
    proc, mod = (processor_en, model_en) if bahasa == "inggris" else (processor_id, model_id)
    inputs = proc(audio, sampling_rate=sr, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = mod(inputs.input_values).logits
    hasil["Wav2Vec2"] = proc.decode(torch.argmax(logits, dim=-1)[0]).strip()

    # 3. MMS
    hasil["MMS"] = mms_pipe(audio, sampling_rate=sr)["text"].strip()

    return hasil

#  EKSEKUSI EKSPERIMEN NOISE
print(f" EKSPERIMEN 2: TRANSKRIPSI AUDIO DENGAN NOISE (Faktor: {NOISE_FACTOR}) \n")

with open(OUTPUT_NOISE_TXT, "w", encoding="utf-8") as f:
    f.write(f"HASIL TRANSKRIPSI AUDIO DENGAN NOISE (Faktor: {NOISE_FACTOR})\n\n")

    for bahasa, file_name in FILES.items():
        print(f"\nMemproses Noise - Bahasa: {bahasa.upper()}")
        f.write(f" BAHASA {bahasa.upper()} \n")

        # 1. Load audio bersih
        audio_clean, sr = load_audio(file_name)

        # 2. Tambahkan noise ke audio tersebut
        audio_noisy = add_noise(audio_clean, noise_factor=NOISE_FACTOR)

        # 3. Jalankan inferensi menggunakan fungsi yang baru dibuat
        transkripsi_noisy = run_pipeline_audio(bahasa, audio_noisy, sr)

        print(f"{'Model':<12} | {'WER':<8} | {'CER':<8} | {'Akurasi':<8}")
        print("-" * 45)

        for model_name, teks in transkripsi_noisy.items():
            # Simpan hasil teks mentah ke file TXT baru
            f.write(f"{model_name}:\n{teks}\n\n")

            # Evaluasi performa model yang terkena noise
            wer, cer, acc = evaluate_text(GROUND_TRUTH[bahasa], teks)
            print(f"{model_name:<12} | {wer:.2%}  | {cer:.2%}  | {acc:.2f}%")

print(f"\nHasil teks khusus noise disimpan di: {OUTPUT_NOISE_TXT}")

 EKSPERIMEN 2: TRANSKRIPSI AUDIO DENGAN NOISE (Faktor: 0.02) 


Memproses Noise - Bahasa: INGGRIS
Model        | WER      | CER      | Akurasi 
---------------------------------------------
Whisper      | 71.43%  | 48.92%  | 28.57%
Wav2Vec2     | 81.51%  | 38.92%  | 20.17%
MMS          | 30.25%  | 7.08%  | 71.43%

Memproses Noise - Bahasa: INDONESIA
Model        | WER      | CER      | Akurasi 
---------------------------------------------
Whisper      | 55.04%  | 19.96%  | 53.49%
Wav2Vec2     | 99.22%  | 67.57%  | 0.78%
MMS          | 26.36%  | 5.82%  | 74.42%

Hasil teks khusus noise disimpan di: transkripsi_noise.txt


In [7]:
SPEED_FACTOR = 1.25
OUTPUT_SPEED_TXT = "transkripsi_speed.txt"

#  FUNGSI PENGUBAH KECEPATAN
def change_speed(audio, rate=1.25):
    audio_stretched = librosa.effects.time_stretch(y=audio, rate=rate)
    return audio_stretched

print(f" EKSPERIMEN 3: TRANSKRIPSI AUDIO DENGAN PERUBAHAN KECEPATAN (Rate: {SPEED_FACTOR}x) \n")

with open(OUTPUT_SPEED_TXT, "w", encoding="utf-8") as f:
    f.write(f"HASIL TRANSKRIPSI AUDIO DENGAN PERUBAHAN KECEPATAN (Rate: {SPEED_FACTOR}x)\n\n")

    for bahasa, file_name in FILES.items():
        print(f"\nMemproses Kecepatan - Bahasa: {bahasa.upper()}")
        f.write(f" BAHASA {bahasa.upper()} \n")

        audio_clean, sr = load_audio(file_name)
        audio_speed = change_speed(audio_clean, rate=SPEED_FACTOR)
        transkripsi_speed = run_pipeline_audio(bahasa, audio_speed, sr)

        print(f"{'Model':<12} | {'WER':<8} | {'CER':<8} | {'Akurasi':<8}")
        print("-" * 45)

        for model_name, teks in transkripsi_speed.items():
            # Simpan hasil teks mentah ke file TXT baru
            f.write(f"{model_name}:\n{teks}\n\n")

            # Evaluasi performa model yang diuji dengan kecepatan baru
            wer, cer, acc = evaluate_text(GROUND_TRUTH[bahasa], teks)
            print(f"{model_name:<12} | {wer:.2%}  | {cer:.2%}  | {acc:.2f}%")

print(f"\nHasil teks khusus kecepatan disimpan di: {OUTPUT_SPEED_TXT}")

 EKSPERIMEN 3: TRANSKRIPSI AUDIO DENGAN PERUBAHAN KECEPATAN (Rate: 1.25x) 


Memproses Kecepatan - Bahasa: INGGRIS
Model        | WER      | CER      | Akurasi 
---------------------------------------------
Whisper      | 68.91%  | 51.85%  | 34.45%
Wav2Vec2     | 81.51%  | 39.69%  | 24.37%
MMS          | 36.97%  | 11.23%  | 63.87%

Memproses Kecepatan - Bahasa: INDONESIA
Model        | WER      | CER      | Akurasi 
---------------------------------------------
Whisper      | 95.35%  | 66.63%  | 4.65%
Wav2Vec2     | 82.95%  | 35.55%  | 19.38%
MMS          | 32.56%  | 7.69%  | 69.77%

Hasil teks khusus kecepatan disimpan di: transkripsi_speed.txt
